# 01 · Constellations and bit mapping

A digital modulator turns groups of `k` bits into one of `M = 2^k` complex
**symbols**, the points of a constellation.  This notebook covers:

1. the standard families (QAM, PSK, PAM) and their normalization;
2. **Gray labeling**, the bit assignment that turns most symbol errors into
   single bit errors;
3. mapping and hard demapping;
4. **rotational symmetry**, the property that later forces carrier recovery
   to resolve an ambiguity;
5. **probabilistic shaping**: sending inner points more often than outer
   ones.

Soft demapping (LLRs) needs a noise model and is covered in
[06 · AWGN and performance metrics](06-awgn-and-performance-metrics.ipynb).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import Constellation
from commkit.backend import to_device

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()

## 1. Families

A `Constellation` is an immutable value object: points, bit labels and an
optional prior.  The factories build the usual families at **unit average
power**, `E[|s|²] = 1`, so a given Es/N0 means the same noise level for every
format.

| family | points | bits/symbol | typical use |
|---|---|---|---|
| `Constellation.psk(M)` | M points on a circle | log2 M | constant envelope: QPSK, 8PSK |
| `Constellation.qam(M)` | square grid (cross for odd k) | log2 M | coherent optical and wireless |
| `Constellation.pam(M)` | M real levels | log2 M | intensity modulation (`unipolar=True`) |

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, c in zip(
    axes,
    [
        Constellation.psk(8),
        Constellation.qam(16),
        Constellation.qam(32),
        Constellation.qam(64),
    ],
):
    ck.plotting.plot_ideal_constellation(c, ax=ax, title=repr(c))
plt.show()

for c in [Constellation.psk(8), Constellation.qam(16), Constellation.pam(4)]:
    print(f"{c!r:28s} k = {c.bits_per_symbol}, E|s|^2 = {c.power():.3f}")

## 2. Gray labeling

The bit labels printed next to each point are a **Gray code**: points at the
minimum distance differ in exactly one bit.  At useful SNRs almost every
symbol error lands on a nearest neighbour, so a symbol error costs one bit
error, and `BER ≈ SER / k`.

Check it directly: for every pair of nearest neighbours, count the bits in
which their labels differ.

In [ ]:
c = Constellation.qam(16)
d = np.abs(c.points[:, None] - c.points[None, :])
d_min = d[d > 0].min()
i, j = np.nonzero(np.isclose(d, d_min))
hamming = np.sum(c.bit_labels[i] != c.bit_labels[j], axis=1)
print(f"{i.size // 2} nearest-neighbour pairs, label distances: {np.unique(hamming)}")

# The 1-D building block: a binary-reflected Gray code.
print("gray_code(3) =", ck.mapping.gray_code(3))

## 3. Mapping and hard demapping

`Constellation.map(bits)` groups the bits by `k` (most significant first)
and returns the symbols; `Constellation.demap(symbols)` returns the bits of
the **nearest** point (minimum Euclidean distance, the maximum-likelihood
decision under Gaussian noise).  Both run on the device of their input.
The functions `map_bits` and `demap_symbols_hard` are the same operations
as free functions; `demap_symbols_hard` also takes a `Signal`.

In [ ]:
rng = np.random.default_rng(0)
bits = rng.integers(0, 2, 4 * 8, dtype=np.uint8)
symbols = c.map(bits)
print("bits    :", bits[:8], "...")
print("symbols :", np.round(symbols[:2], 3), "...")
print("round trip exact:", np.array_equal(c.demap(symbols), bits))

# A small perturbation does not change the decision; a large one does.
noisy = symbols + 0.1 * (
    rng.standard_normal(symbols.size) + 1j * rng.standard_normal(symbols.size)
)
print("bit errors after small noise:", int(np.sum(c.demap(noisy) != bits)))

## 4. Rotational symmetry

Rotating a square QAM by 90° maps the constellation onto itself, and M-PSK
is invariant under rotations by 360°/M.  A receiver that estimates the
carrier phase from the data alone (blind) cannot tell these rotations apart:
the decisions are equally good, but the bits are wrong.  Carrier recovery
resolves the ambiguity with known symbols or differential coding (see
[11 · Blind carrier-phase recovery](11-blind-carrier-phase-recovery.ipynb)).

In [ ]:
for c_ in [Constellation.qam(16), Constellation.psk(8), Constellation.pam(4)]:
    print(f"{c_!r:24s} rotational symmetry: {c_.rotational_symmetry}")

rotated = 1j * symbols  # a 90 degree carrier-phase error
same = np.allclose(
    np.sort_complex(np.round(1j * c.points, 6)), np.sort_complex(np.round(c.points, 6))
)
print("16-QAM rotated by 90 degrees is the same set of points:", same)
print(
    "bit errors after a 90 degree rotation:",
    int(np.sum(c.demap(rotated) != bits)),
    "of",
    bits.size,
)

## 5. Probabilistic shaping

With uniform probabilities a 64-QAM symbol carries 6 bits.  Sending the
inner points more often lowers the average energy for the same minimum
distance, which recovers part of the 1.53 dB gap between square QAM and the
Shannon capacity.  The standard prior is the **Maxwell-Boltzmann**
distribution,

$$P(s_i) \propto e^{-\nu |s_i|^2},$$

with `ν` set so that the entropy `H = -Σ P log2 P` hits a target rate.
`Constellation.shaped(entropy=...)` (or `nu=...`) returns a shaped copy,
rescaled to unit average power again.  `generate` turns random bits into
shaped symbols with a constant-composition distribution matcher (CCDM,
in `commkit.coding`): each channel's record holds every point exactly as
often as the prior says, rounded to `1/N`, and its reference keeps both
the labels on the air and the matcher's input bits.

In [ ]:
c64 = Constellation.qam(64)
ps = c64.shaped(entropy=5.0)
print(
    ps,
    f"| nu = {ck.mapping.optimal_nu(c64, entropy=5.0):.3f} | power = {ps.power():.3f}",
)

tx = ck.generate(ps, num_symbols=2**16, symbol_rate=1e9, rng=1, device=DEVICE)
s = to_device(tx.samples, "cpu")
idx = np.argmin(np.abs(s[:, None] - ps.points[None, :]), axis=1)
freq = np.bincount(idx, minlength=ps.order) / s.size
print(
    f"empirical entropy = {-np.sum(freq[freq > 0] * np.log2(freq[freq > 0])):.3f} bits"
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ck.plotting.plot_constellation(tx, ax=axes[0], title="PS-64QAM, H = 5 bits")
axes[1].scatter(np.abs(ps.points), ps.pmf, label="Maxwell-Boltzmann prior")
axes[1].scatter(np.abs(ps.points), freq, marker="x", label="generated")
axes[1].set_xlabel("|s|")
axes[1].set_ylabel("probability")
axes[1].legend()
plt.show()

**Summary**

* Factories give unit-power constellations with Gray labels; functions take
  the object, never a string.
* `map` / `demap` are the hard mapping pair; nearest-point decisions are ML
  for Gaussian noise.
* The rotational symmetry (4 for QAM) is the ambiguity blind receivers must
  resolve later.
* A shaped constellation keeps unit power and carries `entropy` bits per
  symbol; `generate` reaches it from bits through a distribution matcher.

**Next:** [02 · Spectral analysis](02-spectral-analysis.ipynb).